# TV2 — Kỹ thuật Đặc trưng & Đóng băng `features_v1` (Spark MLlib)

Chạy các cell từ trên xuống. Đọc dữ liệu chia sẵn `tv1_v1` của TV1, gọi module `src/features.py` để tạo bảng đặc trưng `features_v1` (31 cột trên Train, 30 cột trên Val/Test), kiểm thử chống rò rỉ nhãn, chạy smoke test `make_pipeline_lr` / `make_pipeline_gbt(max_bins=300)` và đồng bộ lên Google Drive.

## 1. Môi trường Spark (Java 17 & PySpark 4.0.1)

In [ ]:
# CELL 1 — Cài đặt Java 17 và PySpark 4.0.1 (chạy ngay 1 lần không cần Restart)
import os, sys, subprocess, importlib, importlib.metadata
from pathlib import Path

if 'google.colab' in sys.modules:
    if not list(Path('/usr/lib/jvm').glob('java-17-openjdk-*')):
        subprocess.run(['apt-get', '-qq', 'update'], check=True)
        subprocess.run(['apt-get', '-qq', 'install', '-y', 'openjdk-17-jdk-headless'], check=True)
    java17 = sorted(Path('/usr/lib/jvm').glob('java-17-openjdk-*'))[0]
    os.environ['JAVA_HOME'] = str(java17)
    os.environ['PATH'] = f"{java17 / 'bin'}:{os.environ['PATH']}"
    if importlib.metadata.version('pyspark') != '4.0.1':
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyspark==4.0.1'])
        for m in [k for k in sys.modules if k == 'pyspark' or k.startswith(('pyspark.', 'py4j'))]:
            sys.modules.pop(m, None)
        importlib.invalidate_caches()

import pyspark
os.environ['PYSPARK_PYTHON'] = sys.executable
print('Python:', sys.version.split()[0], '| PySpark:', pyspark.__version__)
assert pyspark.__version__ == '4.0.1', 'Vui lòng dùng PySpark 4.0.1'

## 2. Kết nối Drive, clone repo và kiểm tra dữ liệu `tv1_v1`

In [ ]:
# CELL 2 — Mount Drive, tải repo chứa src/features.py và copy splits về đĩa cục bộ Colab
import hashlib, json, shutil
from google.colab import drive
drive.mount('/content/drive')

REPO = Path('/content/BDA-Nhom_8')
if not (REPO / 'src' / 'features.py').is_file():
    subprocess.run(['git', 'clone', 'https://github.com/BAC164/BDA-Nhom_8.git', str(REPO)], check=True)
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / 'src'))

DRIVE_ROOT = Path('/content/drive/MyDrive/BigData_Topic20')
TV1_DIR = DRIVE_ROOT / '01_data_frozen' / 'tv1_v1'
FEATURES_ON_DRIVE = DRIVE_ROOT / '01_data_frozen' / 'features_v1'
SPLITS_LOCAL = Path('/content/splits')
FEATURES_LOCAL = Path('/content/features_v1')

def compute_folder_checksums(folder: Path) -> dict:
    out = {}
    for p in sorted(folder.rglob('*')):
        if p.is_file():
            h = hashlib.sha256()
            with p.open('rb') as f:
                for blk in iter(lambda: f.read(1024 * 1024), b''):
                    h.update(blk)
            out[p.relative_to(folder).as_posix()] = h.hexdigest()
    return out

assert TV1_DIR.is_dir(), f'Chưa thấy {TV1_DIR} trên Drive. Kiểm tra shortcut BigData_Topic20.'
manifest = json.loads((TV1_DIR / 'manifest.json').read_text(encoding='utf-8'))
expected_hashes = json.loads((TV1_DIR / 'split_checksums.json').read_text(encoding='utf-8'))
for name in ('train', 'val', 'test'):
    dst = SPLITS_LOCAL / name
    if not dst.is_dir():
        shutil.copytree(TV1_DIR / 'splits' / name, dst)
    assert compute_folder_checksums(dst) == expected_hashes[name], f'Sai checksum ở {name}!'
print('Đã xác minh checksum tv1_v1:', manifest['counts'])

## 3. Khởi tạo Spark và xây dựng bảng đặc trưng `features_v1`

In [ ]:
# CELL 3 — Đọc splits chuẩn TV1 và chạy build_features_v1 từ src/features.py
from pyspark.sql import SparkSession
from src.features import (
    build_features_v1, save_features_and_lookups, verify_no_label_leakage,
    make_pipeline_lr, make_pipeline_gbt
)

spark = (
    SparkSession.builder
    .appName('BDA_Nhom8_TV2_Features_v1')
    .master('local[2]')
    .config('spark.driver.memory', '4g')
    .config('spark.driver.maxResultSize', '2g')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.sql.autoBroadcastJoinThreshold', '20971520')
    .config('spark.sql.session.timeZone', 'UTC')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')

train_raw = spark.read.parquet(str(SPLITS_LOCAL / 'train'))
val_raw   = spark.read.parquet(str(SPLITS_LOCAL / 'val'))
test_raw  = spark.read.parquet(str(SPLITS_LOCAL / 'test'))

train_fe, val_fe, test_fe, artifacts = build_features_v1(
    train_raw, val_raw, test_raw,
    smoothing_m=30.0, min_count_airport=100, min_count_route=50,
    fit_density_on_train_only=True
)
train_fe.cache()
val_fe.cache()
train_fe.printSchema()
train_fe.select('row_id', 'Airline', 'route', 'Length', 'Length_invalid', 'hour', 'hour_cat',
                'density_from', 'AirportFrom_g', 'log_freq_route', 'te_route', 'te_flight', 'Delay').show(5, truncate=False)

## 4. Kiểm thử chống rò rỉ nhãn (`verify_no_label_leakage`)

In [ ]:
# CELL 4 — Đối chiếu công thức Val, OOF Train (gồm oof_prior) và kiểm tra AUC đơn biến te_route
leakage_report = verify_no_label_leakage(
    train_raw=train_raw, train_fe=train_fe, val_fe=val_fe,
    prior=artifacts['prior'], m=artifacts['smoothing_m']
)
assert 'PASSED' in leakage_report['status'], 'Kiểm thử rò rỉ nhãn chưa đạt!'

## 5. Lưu `features_v1`, `lookup_maps/` và chạy Smoke Test 2 Pipeline

In [ ]:
# CELL 5 — Ghi Parquet + lookup_maps + manifest/checksums và kiểm thử make_pipeline_lr / make_pipeline_gbt
save_features_and_lookups(train_fe, val_fe, test_fe, artifacts, output_base_dir=str(FEATURES_LOCAL))

v1_parts = {k: spark.read.parquet(str(FEATURES_LOCAL / k)) for k in ('train', 'val', 'test')}
v1_counts = {k: df.count() for k, df in v1_parts.items()}
assert v1_counts == {'train': 376749, 'val': 81606, 'test': 81028}

v1_checksums = {k: compute_folder_checksums(FEATURES_LOCAL / k) for k in ('train', 'val', 'test')}
v1_manifest = {
    'version': 'features_v1', 'tag': 'data-freeze-v1', 'parent_data': 'tv1_v1',
    'counts': v1_counts, 'schemas': {k: df.schema.jsonValue() for k, df in v1_parts.items()},
    'prior_train_full': artifacts['prior'], 'smoothing_m': artifacts['smoothing_m'],
    'min_count_airport': artifacts['rare_maps']['min_count_airport'],
    'min_count_route': artifacts['rare_maps']['min_count_route'],
    'leakage_check': leakage_report['status'],
}
(FEATURES_LOCAL / 'manifest.json').write_text(json.dumps(v1_manifest, ensure_ascii=False, indent=2), encoding='utf-8')
(FEATURES_LOCAL / 'split_checksums.json').write_text(json.dumps(v1_checksums, indent=2), encoding='utf-8')

tr_sample = v1_parts['train'].sample(False, 0.05, seed=42)
val_sample = v1_parts['val'].sample(False, 0.05, seed=42)
print('Smoke test LR rows:', make_pipeline_lr(max_iter=10).fit(tr_sample).transform(val_sample).count())
print('Smoke test GBT rows:', make_pipeline_gbt(max_bins=300, max_iter=5, max_depth=4).fit(tr_sample).transform(val_sample).count())

## 6. Đồng bộ `features_v1` lên Google Drive của nhóm

In [ ]:
# CELL 6 — Xóa bản cũ (nếu có), copy lên Drive và xác minh lại SHA-256 checksum
import time

if FEATURES_ON_DRIVE.exists():
    shutil.rmtree(FEATURES_ON_DRIVE)
shutil.copytree(FEATURES_LOCAL, FEATURES_ON_DRIVE)
time.sleep(3)

for split_name in ('train', 'val', 'test'):
    assert compute_folder_checksums(FEATURES_ON_DRIVE / split_name) == v1_checksums[split_name], f'Lỗi checksum ở {split_name}!'
print(f'[DRIVE SUCCESS] Đã đồng bộ và xác minh checksum features_v1 tại: {FEATURES_ON_DRIVE}')